# Session 1 - 21.04.2026.

In [1]:
import numpy as np
import torch

In [2]:
class Tensor:
    def __init__(self, data: np.ndarray | list, prev: set = None):
        self.data: np.ndarray = data if isinstance(data, np.ndarray) else np.array(data)
        self.grad: np.ndarray = np.zeros_like(data)
        self._backward = lambda: None
        self._prev: set = set() if prev is None else prev
        # requires_grad boolean - if it is not true pytorch wont allow backward call
        # additionally pytorch does not allow grads for ints, only floats and complex
        # device
        # dtype

    
    def __add__(self, other) -> Tensor:
        out: Tensor = Tensor(self.data + other.data, {self, other})
        def backward() -> None:
            self.grad += out.grad * 1
            other.grad += out.grad * 1
        out._backward = backward
        return out


    def __matmul__(self, other) -> Tensor:
        # assert self.data.shape[-1] == other.data.shape[-2] / "Error dimensions not match"
        out: Tensor = Tensor(self.data @ other.data, {self, other})
        def backward() -> None:
            # X @ W = Z
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad
        out._backward = backward
        return out

    
    def backward(self) -> None:
        # This is my initialization step
        self.grad: np.ndarray = np.ones_like(self.data)
        visited: set = set()
        topo: list[Tensor] = []
        def build_topo(t: Tensor) -> None:
            if t not in visited:
                visited.add(t)
                for p in t._prev:
                    build_topo(p)
                topo.append(t)
        build_topo(self)
        for t in reversed(topo):
            t._backward()


In [3]:
a = Tensor(np.arange(0, 10), set())
b = Tensor(np.arange(0, 10), set())
c = a + b
c.data

array([ 0,  2,  4,  6,  8, 10, 12, 14, 16, 18])

In [4]:
c.backward()

In [5]:
a.grad

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [6]:
c.grad

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [7]:
a = Tensor(np.arange(0, 9).reshape(3,3), set())
b = Tensor(np.arange(0, 9).reshape(3,3), set())
c = a @ b
c.data

array([[ 15,  18,  21],
       [ 42,  54,  66],
       [ 69,  90, 111]])

In [8]:
c.backward()

In [9]:
a.grad

array([[ 3, 12, 21],
       [ 3, 12, 21],
       [ 3, 12, 21]])

In [10]:
b.grad

array([[ 9,  9,  9],
       [12, 12, 12],
       [15, 15, 15]])

In [11]:
a_t = torch.tensor(np.arange(0, 9).reshape(3,3).astype(np.float32), requires_grad=True)
b_t = torch.tensor(np.arange(0, 9).reshape(3,3).astype(np.float32), requires_grad=True)
c_t = a_t @ b_t
c_t

tensor([[ 15.,  18.,  21.],
        [ 42.,  54.,  66.],
        [ 69.,  90., 111.]], grad_fn=<MmBackward0>)

In [12]:
a_t

tensor([[0., 1., 2.],
        [3., 4., 5.],
        [6., 7., 8.]], requires_grad=True)

In [13]:
c_t

tensor([[ 15.,  18.,  21.],
        [ 42.,  54.,  66.],
        [ 69.,  90., 111.]], grad_fn=<MmBackward0>)

In [14]:
c_t.data

tensor([[ 15.,  18.,  21.],
        [ 42.,  54.,  66.],
        [ 69.,  90., 111.]])

In [15]:
c_t.backward(torch.ones_like(c_t.data))

In [16]:
# PyTorch wont let me call c_t.backward() because c_t is not a scalar but rather a matrix in this case so I have to provide it with explicit starting point for backward()

In [17]:
help(torch.Tensor.backward)

Help on function backward in module torch._tensor:

backward(
    self,
    gradient=None,
    retain_graph=None,
    create_graph=False,
    inputs=None
)
    Computes the gradient of current tensor wrt graph leaves.

    The graph is differentiated using the chain rule. If the tensor is
    non-scalar (i.e. its data has more than one element) and requires
    gradient, the function additionally requires specifying a ``gradient``.
    It should be a tensor of matching type and shape, that represents
    the gradient of the differentiated function w.r.t. ``self``.

    This function accumulates gradients in the leaves - you might need to zero
    ``.grad`` attributes or set them to ``None`` before calling it.
    See :ref:`Default gradient layouts<default-grad-layouts>`
    for details on the memory layout of accumulated gradients.

    .. note::

        If you run any forward ops, create ``gradient``, and/or call ``backward``
        in a user-specified CUDA stream context, see
     

In [18]:
a_t.grad

tensor([[ 3., 12., 21.],
        [ 3., 12., 21.],
        [ 3., 12., 21.]])

In [19]:
help(torch.Tensor)

Help on class Tensor in module torch:

class Tensor(torch._C.TensorBase)
 |  Method resolution order:
 |      Tensor
 |      torch._C.TensorBase
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __abs__ = abs(self, /)
 |
 |  __array__(self, dtype=None) from torch._tensor.Tensor
 |
 |  __array_wrap__(self, array) from torch._tensor.Tensor
 |      # Wrap Numpy array again in a suitable tensor when done, to support e.g.
 |      # `numpy.sin(tensor) -> tensor` or `numpy.greater(tensor, 0) -> ByteTensor`
 |
 |  __contains__(self, element: Any, /) -> bool from torch._tensor.Tensor
 |      Check if `element` is present in tensor
 |
 |      Args:
 |          element (Tensor or scalar): element to be checked
 |              for presence in current tensor"
 |
 |  __deepcopy__(self, memo) from torch._tensor.Tensor
 |
 |  __dir__(self) from torch._tensor.Tensor
 |      Default dir() implementation.
 |
 |  __dlpack__(
 |      self,
 |      *,
 |      stream: Any | None = -1,
 |      max_v

In [20]:
# Ok I def need some allclose() in order to test how far are my impls
help(torch.allclose)

Help on built-in function allclose in module torch:

allclose(...)
    allclose(input: Tensor, other: Tensor, rtol: float = 1e-05, atol: float = 1e-08, equal_nan: bool = False) -> bool

    This function checks if :attr:`input` and :attr:`other` satisfy the condition:

    .. math::
        \lvert \text{input}_i - \text{other}_i \rvert \leq \texttt{atol} + \texttt{rtol} \times \lvert \text{other}_i \rvert

    elementwise, for all elements of :attr:`input` and :attr:`other`. The behaviour of this function is analogous to
    `numpy.allclose <https://numpy.org/doc/stable/reference/generated/numpy.allclose.html>`_

    Args:
        input (Tensor): first tensor to compare
        other (Tensor): second tensor to compare
        atol (float, optional): absolute tolerance. Default: 1e-08
        rtol (float, optional): relative tolerance. Default: 1e-05
        equal_nan (bool, optional): if ``True``, then two ``NaN`` s will be considered equal. Default: ``False``

    Example::

        >

In [21]:
torch.allclose(torch.tensor([1.1]), torch.tensor([1.1]), atol=1e-5)

True

In [22]:
torch.allclose(torch.tensor([1.10000]), torch.tensor([1.10001]), atol=1e-5)

True

In [23]:
torch.allclose(torch.tensor([1.1001]), torch.tensor([1.1000]), atol=1e-5)

False

In [ ]:
# This will become my first test in a bit, check what atol is reasonable 

from typing import Callable


def test(torch_fn: Callable, picograd_fn: Callable, atol: float) -> bool:
    return True

In [25]:
help(np.random.rand)

Help on method rand in module numpy.random:

rand(*args) method of numpy.random.mtrand.RandomState instance
    rand(d0, d1, ..., dn)

    Random values in a given shape.

    .. note::
        This is a convenience function for users porting code from Matlab,
        and wraps `random_sample`. That function takes a
        tuple to specify the size of the output, which is consistent with
        other NumPy functions like `numpy.zeros` and `numpy.ones`.

    Create an array of the given shape and populate it with
    random samples from a uniform distribution
    over ``[0, 1)``.

    Parameters
    ----------
    d0, d1, ..., dn : int, optional
        The dimensions of the returned array, must be non-negative.
        If no argument is given a single Python float is returned.

    Returns
    -------
    out : ndarray, shape ``(d0, d1, ..., dn)``
        Random values.

    See Also
    --------
    random

    Examples
    --------
    >>> np.random.rand(3,2)
    array([[ 0.1402247

In [26]:
def set_linear(in_dim: int, out_dim: int, use_bias: bool) -> dict[str, Tensor]:
    weights: Tensor = Tensor(np.random.rand((in_dim, out_dim))) # 
    bias: Tensor | None = None
    if use_bias:
        bias = Tensor(np.random.rand(out_dim))
    return {"weights": weights, "bias": bias}

In [27]:
help(np.ones)

Help on function ones in module numpy:

ones(shape, dtype=None, order='C', *, device=None, like=None)
    Return a new array of given shape and type, filled with ones.

    Parameters
    ----------
    shape : int or sequence of ints
        Shape of the new array, e.g., ``(2, 3)`` or ``2``.
    dtype : data-type, optional
        The desired data-type for the array, e.g., `numpy.int8`.  Default is
        `numpy.float64`.
    order : {'C', 'F'}, optional, default: C
        Whether to store multi-dimensional data in row-major
        (C-style) or column-major (Fortran-style) order in
        memory.
    device : str, optional
        The device on which to place the created array. Default: None.
        For Array-API interoperability only, so must be ``"cpu"`` if passed.

        .. versionadded:: 2.0.0
    like : array_like, optional
            Reference object to allow the creation of arrays which are not
            NumPy arrays. If an array-like passed in as ``like`` supports
  

In [28]:
def set_linear_test(in_dim: int, out_dim: int, use_bias: bool) -> dict[str, Tensor]:
    weights: Tensor = Tensor(np.ones((in_dim, out_dim))) # 
    bias: Tensor | None = None
    if use_bias:
        bias = Tensor(np.ones(out_dim))
    return {"weights": weights, "bias": bias}

In [29]:
x = Tensor(np.arange(0, 15).reshape(3, 5))
linear_layer = set_linear_test(5, 10, False)

if linear_layer['bias']:
    z = x @ linear_layer['weights'] + linear_layer['bias']
else:
    z = x @ linear_layer['weights']

In [30]:
z.data

array([[10., 10., 10., 10., 10., 10., 10., 10., 10., 10.],
       [35., 35., 35., 35., 35., 35., 35., 35., 35., 35.],
       [60., 60., 60., 60., 60., 60., 60., 60., 60., 60.]])

In [31]:
def set_linear_test_torch(in_dim: int, out_dim: int, use_bias: bool) -> dict[str, Tensor]:
    weights: Tensor = torch.ones((in_dim, out_dim)) # 
    bias: Tensor | None = None
    if use_bias:
        bias = torch.ones(out_dim)
    return {"weights": weights, "bias": bias}

In [32]:
x_torch = torch.tensor(np.arange(0, 15).reshape(3, 5).astype(np.float32))
linear_layer_torch = set_linear_test_torch(5, 10, False)

if linear_layer_torch['bias'] is not None:
    z_torch = x_torch @ linear_layer_torch['weights'] + linear_layer_torch['bias']
else:
    z_torch = x_torch @ linear_layer_torch['weights']

In [33]:
z_torch.data


tensor([[10., 10., 10., 10., 10., 10., 10., 10., 10., 10.],
        [35., 35., 35., 35., 35., 35., 35., 35., 35., 35.],
        [60., 60., 60., 60., 60., 60., 60., 60., 60., 60.]])

In [34]:
help(torch.autograd.function)

Help on module torch.autograd.function in torch.autograd:

NAME
    torch.autograd.function - # mypy: allow-untyped-defs

CLASSES
    builtins.object
        FunctionCtx
            BackwardCFunction(torch._C._FunctionBase, FunctionCtx, _HookMixin)
    builtins.type(builtins.object)
        FunctionMeta
    torch._C._FunctionBase(builtins.object)
        BackwardCFunction(torch._C._FunctionBase, FunctionCtx, _HookMixin)
    _SingleLevelFunction(torch._C._FunctionBase, FunctionCtx, _HookMixin)
        Function
            InplaceFunction
            NestedIOFunction

    class BackwardCFunction(torch._C._FunctionBase, FunctionCtx, _HookMixin)
     |  This class is used for internal autograd work. Do not use.
     |
     |  Method resolution order:
     |      BackwardCFunction
     |      torch._C._FunctionBase
     |      FunctionCtx
     |      _HookMixin
     |      builtins.object
     |
     |  Methods defined here:
     |
     |  apply(self, *args)
     |      Apply method used wh

# Session 2 - 22.04.2026.

In [35]:
# I defintely need to as dtype asap because it is causing me problems all over the place
# I also assume that implicite broadcast in forawrd pass will cause problems in backward pass since I want accumulate grads over those expanded dims

In [36]:
x = Tensor(np.arange(0, 15).reshape(3, 5).astype(np.float32))
linear_layer = set_linear_test(5, 10, True)

if linear_layer['bias']:
    z = x @ linear_layer['weights'] + linear_layer['bias']
else:
    z = x @ linear_layer['weights']

In [ ]:
# z.backward() this will cause an error so I will comment it out

ValueError: non-broadcastable output operand with shape (10,) doesn't match the broadcast shape (3,10)

In [38]:
linear_layer['bias'].data

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [40]:
# Prior to wrestling with broadcasting could I expect from user to be enter correct shapes all the time or I can omit bias from my first versions of layer

In [39]:
help(torch.Tensor.sum)

Help on method descriptor sum:

sum(...) unbound torch._C.TensorBase method
    sum(dim=None, keepdim=False, dtype=None) -> Tensor

    See :func:`torch.sum`



In [41]:
# This shall be the first reduction op that we implement

# Session 3 24.04.2026.

In [42]:
help(torch.nn.Module)

Help on class Module in module torch.nn.modules.module:

class Module(builtins.object)
 |  Module(*args: Any, **kwargs: Any) -> None
 |
 |  Base class for all neural network modules.
 |
 |  Your models should also subclass this class.
 |
 |  Modules can also contain other Modules, allowing them to be nested in
 |  a tree structure. You can assign the submodules as regular attributes::
 |
 |      import torch.nn as nn
 |      import torch.nn.functional as F
 |
 |
 |      class Model(nn.Module):
 |          def __init__(self) -> None:
 |              super().__init__()
 |              self.conv1 = nn.Conv2d(1, 20, 5)
 |              self.conv2 = nn.Conv2d(20, 20, 5)
 |
 |          def forward(self, x):
 |              x = F.relu(self.conv1(x))
 |              return F.relu(self.conv2(x))
 |
 |  Submodules assigned in this way will be registered, and will also have their
 |  parameters converted when you call :meth:`to`, etc.
 |
 |  .. note::
 |      As per the example above, an ``__init_

In [ ]:
# I want to avoid clojure and I want to create Function and Ctx like Pytorch

In [ ]:
class Context:
    def __init__(self):
        self.tensors: list[np.ndarray] = []

    def save_for_backward(self):
        pass
    

In [ ]:
# Base class from which all ops/funcs will inherit
# I will use it to substitute intially written clojure that gets saved to _backward attribute of a tensor 
class Funtion:
    def __init__():
        pass
    

In [49]:
from src.nn.layer.linear import Linear
from src.nn.module import Module


class NeuralNetwork(Module):
    def __init__(self) -> None:
        self.layer1 = Linear(3, 5)
        self.layer2 = Linear(5, 3)
        
    def forward(self, x: Tensor) -> Tensor:
        return self.layer2(self.layer1(x).relu()).relu()
        
    def __call__(self, x: Tensor) -> Tensor:
        return self.forward(x)

In [50]:
net = NeuralNetwork()

In [53]:
for params in net.parameters():
    print(params.data)


[[ 0.26858192 -0.23928708 -0.01509905  1.67460413  0.15492878]
 [ 2.36075894 -1.01869861 -0.29446908 -0.12287817 -0.45691666]
 [ 1.34377104  0.75773619 -1.78754021 -0.65471141 -0.77961669]]
[[-1.1952313  -1.83947455  0.52513114]
 [ 0.93285208 -1.25871695 -2.64768824]
 [ 0.32733419  0.11589465 -0.68808832]
 [-1.08903349  0.4966784  -0.65776744]
 [-0.6885132  -3.31858702  0.4401051 ]]


In [ ]:
# Module paramsters fn works

In [54]:
help(torch.nn.Linear.reset_parameters)

Help on function reset_parameters in module torch.nn.modules.linear:

reset_parameters(self) -> None
    Resets parameters based on their initialization used in ``__init__``.



In [57]:
np.mean(np.random.randn(1, 1000))

np.float64(-0.03431782164763225)

# Session 4 5.5.2026.

In [2]:
help(torch.autograd.Function)

Help on class Function in module torch.autograd.function:

class Function(_SingleLevelFunction)
 |  Function(*args, **kwargs)
 |
 |  Base class to create custom `autograd.Function`.
 |
 |  To create a custom `autograd.Function`, subclass this class and implement
 |  the :meth:`forward` and :meth:`backward` static methods. Then, to use your custom
 |  op in the forward pass, call the class method ``apply``. Do not call
 |  :meth:`forward` directly.
 |
 |  To ensure correctness and best performance, make sure you are calling the
 |  correct methods on ``ctx`` and validating your backward function using
 |  :func:`torch.autograd.gradcheck`.
 |
 |  See :ref:`extending-autograd` for more details on how to use this class.
 |
 |  Examples::
 |
 |      >>> # xdoctest: +REQUIRES(env:TORCH_DOCTEST_AUTOGRAD)
 |      >>> class Exp(Function):
 |      >>>     @staticmethod
 |      >>>     def forward(ctx, i):
 |      >>>         result = i.exp()
 |      >>>         ctx.save_for_backward(result)
 |  

In [3]:
help(torch.Tensor.grad_fn)

Help on getset descriptor torch._C.TensorBase.grad_fn:

grad_fn



In [4]:
help(torch.Tensor)

Help on class Tensor in module torch:

class Tensor(torch._C.TensorBase)
 |  Method resolution order:
 |      Tensor
 |      torch._C.TensorBase
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __abs__ = abs(self, /)
 |
 |  __array__(self, dtype=None) from torch._tensor.Tensor
 |
 |  __array_wrap__(self, array) from torch._tensor.Tensor
 |      # Wrap Numpy array again in a suitable tensor when done, to support e.g.
 |      # `numpy.sin(tensor) -> tensor` or `numpy.greater(tensor, 0) -> ByteTensor`
 |
 |  __contains__(self, element: Any, /) -> bool from torch._tensor.Tensor
 |      Check if `element` is present in tensor
 |
 |      Args:
 |          element (Tensor or scalar): element to be checked
 |              for presence in current tensor"
 |
 |  __deepcopy__(self, memo) from torch._tensor.Tensor
 |
 |  __dir__(self) from torch._tensor.Tensor
 |      Default dir() implementation.
 |
 |  __dlpack__(
 |      self,
 |      *,
 |      stream: Any | None = -1,
 |      max_v

In [5]:
help(torch.autograd.function.FunctionCtx)

Help on class FunctionCtx in module torch.autograd.function:

class FunctionCtx(builtins.object)
 |  # Formerly known as: _ContextMethodMixin
 |
 |  Methods defined here:
 |
 |  mark_dirty(self, *args: torch.Tensor)
 |      Mark given tensors as modified in an in-place operation.
 |
 |      This should be called at most once, in either the :func:`setup_context`
 |      or :func:`forward` methods, and all arguments should be inputs.
 |
 |      Every tensor that's been modified in-place in a call to :func:`forward`
 |      should be given to this function, to ensure correctness of our checks.
 |      It doesn't matter whether the function is called before or after
 |      modification.
 |
 |      Examples::
 |          >>> # xdoctest: +REQUIRES(env:TORCH_DOCTEST_AUTOGRAD)
 |          >>> class Inplace(Function):
 |          >>>     @staticmethod
 |          >>>     def forward(ctx, x):
 |          >>>         x_npy = x.numpy() # x_npy shares storage with x
 |          >>>         x_npy +

In [6]:
help(torch.autograd.Function.apply)

Help on method apply in module torch.autograd.function:

apply(*args, **kwargs) class method of torch.autograd.function.Function



In [9]:
help(torch.Tensor.grad_fn)

Help on getset descriptor torch._C.TensorBase.grad_fn:

grad_fn



In [10]:
a_t: torch.Tensor = torch.tensor([[1, 2, 3], [1, 1, 1]], dtype=torch.float32, requires_grad=True)
b_t: torch.Tensor = torch.tensor([[1, 2, 1], [1, 2, 3], [1, 2, 2]], dtype=torch.float32, requires_grad=True) 
c_t = a_t @ b_t
c_t.backward(torch.ones_like(c_t))

In [16]:
type(c_t)

torch.Tensor